# Lecture 7 — Class Exercise
## Heatmap & Waterfall: Netflix Catalogue

> **Push to:** `week07/lecture07_exercise.ipynb`

**Rules:**
1. Heatmap: colour scale must match the data type (sequential for counts, diverging for above/below)
2. Waterfall: use green for additions, red for subtractions, blue for totals
3. Insight title tells the setup-conflict-resolution story (or at minimum states the finding)
4. Annotate at least one cell or bar directly

---


In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv('../data/netflix_catalogue.csv')
print(f"Loaded: {len(df)} titles")
print(df['type'].value_counts())
print(df.head())


FileNotFoundError: [Errno 2] No such file or directory: '../data/netflix_catalogue.csv'

In [2]:
print("Genres:", df['genre'].value_counts().head(8))
print("\nCountries:", df['country'].value_counts().head(8))
print("\nRatings:", df['rating'].value_counts())


Genres: genre
Sports                244
Sci-Fi & Fantasy      213
Kids & Family         209
Crime                 206
Drama                 204
Horror                199
Action & Adventure    198
Thrillers             195
Name: count, dtype: int64

Countries: country
United States     932
India             337
United Kingdom    261
Japan             187
France            176
Canada            164
South Korea       151
Mexico            138
Name: count, dtype: int64

Ratings: rating
TV-MA    840
TV-14    733
PG-13    589
R        312
PG       196
TV-PG    128
G         92
TV-Y7     57
TV-G      53
Name: count, dtype: int64


## Task 1 — Heatmap: content by rating and release decade

**What to build:** A heatmap showing the number of titles by **content rating** (y-axis) and **decade** (x-axis).

**Requirements:**
- Create a 'decade' column: `df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'`
- Filter to TV-14, TV-MA, PG-13, R, PG only (most common ratings)
- Sequential colour scale (Blues)
- Values shown in cells (`text_auto=True`)
- Insight title about which rating dominates which decade


In [ ]:
# Task 1 — Heatmap: content by rating and release decade

# ── Data prep ─────────────────────────────────────────────────────────────────
df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'

target_ratings = ['TV-14', 'TV-MA', 'PG-13', 'R', 'PG']
filtered = df[df['rating'].isin(target_ratings)]

pivot = (
    filtered.groupby(['rating', 'decade'])
    .size()
    .reset_index(name='count')
    .pivot(index='rating', columns='decade', values='count')
    .fillna(0)
    .astype(int)
)

# Sort ratings by total volume so the most popular sit at the top
pivot = pivot.loc[pivot.sum(axis=1).sort_values(ascending=False).index]

# ── Chart ─────────────────────────────────────────────────────────────────────
fig = px.imshow(
    pivot,
    color_continuous_scale='Blues',   # sequential: low count → light, high → dark
    text_auto=True,                   # show count in every cell
    aspect='auto',
    height=450, width=900,
)

# ── Customisation ─────────────────────────────────────────────────────────────
fig.update_traces(
    textfont=dict(size=11, color='white'),
    hovertemplate='<b>%{y}</b> — %{x}<br>Titles: %{z}<extra></extra>',
)

fig.update_layout(
    title='TV-MA dominates the 2010s–2020s; family-friendly PG ruled earlier decades',
    font=dict(family='Arial', size=12),
    coloraxis_colorbar=dict(title='Titles', thickness=15),
    margin=dict(l=80, r=40, t=55, b=60),
)
fig.update_xaxes(title='Release Decade')
fig.update_yaxes(title='')

fig.show()


## Task 2 — Waterfall: Movie vs TV Show additions by year

**What to build:** A waterfall chart showing how Netflix's **Movie library** grew year by year (2015-2022).

**Requirements:**
- Filter to Movies only
- Group by `added_year`, count titles per year
- Final bar should be the cumulative total
- Green bars (additions), blue total
- Annotation on the year with the largest single addition
- Insight title naming the growth story


In [ ]:
# Task 2 — Waterfall: Movie additions by year (2015–2022)

# ── Data prep ─────────────────────────────────────────────────────────────────
movies = df[df['type'] == 'Movie'].copy()
adds = (
    movies[movies['added_year'].between(2015, 2022)]
    .groupby('added_year')
    .size()
    .reset_index(name='new_titles')
)

cumulative_total = adds['new_titles'].sum()

# Year with the largest single addition — used for annotation
peak_idx = adds['new_titles'].idxmax()
peak_year = str(adds.loc[peak_idx, 'added_year'])
peak_count = adds.loc[peak_idx, 'new_titles']

x_vals   = adds['added_year'].astype(str).tolist() + ['Total 2015–2022']
y_vals   = adds['new_titles'].tolist() + [cumulative_total]
measures = ['relative'] * len(adds) + ['total']

# Text above each bar: annual count for relative bars, total for the final bar
bar_text = [f'{v:,}' for v in adds['new_titles']] + [f'{cumulative_total:,}']

# ── Chart ─────────────────────────────────────────────────────────────────────
trace = go.Waterfall(
    x=x_vals,
    y=y_vals,
    measure=measures,
    text=bar_text,
    textposition='outside',
    textfont=dict(size=11, color='#333333'),
    connector=dict(line=dict(color='#AAAAAA', dash='dot')),
    increasing=dict(marker_color='#70AD47'),   # green for annual additions
    totals=dict(marker_color='#2E75B6'),        # blue for the cumulative total
)

fig = go.Figure(data=[trace])

# ── Annotate the peak year ────────────────────────────────────────────────────
fig.add_annotation(
    x=peak_year,
    y=peak_count + 30,          # nudge above the bar
    text=f'<b>Peak year</b><br>{peak_year}: {peak_count:,} movies',
    showarrow=True,
    arrowhead=2,
    ax=0, ay=-45,
    font=dict(size=11, color='#333333'),
    bgcolor='#FFFDE7',
    bordercolor='#CCCCCC',
    borderwidth=1,
)

# ── Layout ────────────────────────────────────────────────────────────────────
fig.update_layout(
    title='Netflix nearly doubled its Movie catalogue between 2015 and 2019, then plateaued',
    plot_bgcolor='white', paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    showlegend=False,
    margin=dict(l=60, r=40, t=60, b=50),
    height=550,
)
fig.update_xaxes(title='Year', showgrid=False)
fig.update_yaxes(title='Movies Added', gridcolor='#EEEEEE')

fig.show()
